# 🔍 Visual Search Engine — Демо

**Мультимодальный поиск изображений по тексту** с использованием `jina-embeddings-v5-omni-nano` и сравнением с CLIP.

---

## 1. Настройка окружения

In [ ]:
import sys
from pathlib import Path

# Добавляем src в PYTHONPATH
sys.path.insert(0, str(Path.cwd() / "src"))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from config import *
from dataset import prepare_dataset, load_metadata
from embeddings import build_embeddings_jina, build_embeddings_clip
from index import build_or_load_index, search_index, load_index
from search import search_images, display_results
from evaluation import evaluate_model, compare_models, compare_index_types

print('✅ Все модули загружены')

## 2. Подготовка данных

Загружаем подвыборку **Conceptual Captions** (5000 примеров) и скачиваем изображения по URL.

In [ ]:
# Загрузка и подготовка датасета
captions, image_paths = prepare_dataset(force_reload=False)

print(f"📊 Датасет: {len(captions)} изображений")
print(f"📁 Пример пути: {image_paths[0]}")
print(f"📝 Пример подписи: {captions[0][:100]}...")

### Пример изображений из датасета

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    img = Image.open(image_paths[i]).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"ID {i}\n{captions[i][:40]}...", fontsize=9)
    ax.axis('off')
plt.suptitle('Примеры из Conceptual Captions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Создание мультимодальных эмбеддингов

### 3.1 Jina v5-omni-nano (основная модель)

Оптимизации для CPU:
- `modality='vision'` — только vision + text towers
- `truncate_dim=256` — снижение размерности с 768 до 256
- `batch_size=4` — маленький батч для CPU

In [ ]:
%%time
jina_img_emb, jina_txt_emb = build_embeddings_jina(image_paths, captions, force_rebuild=False)

print(f"✅ Jina embeddings созданы")
print(f"   Image: {jina_img_emb.shape} (dtype={jina_img_emb.dtype})")
print(f"   Text:  {jina_txt_emb.shape} (dtype={jina_txt_emb.dtype})")
print(f"   Norm check (first): {np.linalg.norm(jina_img_emb[0]):.4f}")

### 3.2 CLIP ViT-B/32 (бейслайн)

In [ ]:
%%time
clip_img_emb, clip_txt_emb = build_embeddings_clip(image_paths, captions, force_rebuild=False)

print(f"✅ CLIP embeddings созданы")
print(f"   Image: {clip_img_emb.shape} (dtype={clip_img_emb.dtype})")
print(f"   Text:  {clip_txt_emb.shape} (dtype={clip_txt_emb.dtype})")
print(f"   Norm check (first): {np.linalg.norm(clip_img_emb[0]):.4f}")

## 4. Создание векторных индексов (FAISS)

Используем два типа индексов:
- **IndexFlatIP** — точный поиск (brute-force cosine similarity)
- **IndexHNSWFlat** — приближённый поиск (ANN)

In [ ]:
# Jina Flat
jina_flat = build_or_load_index(jina_img_emb, FAISS_FLAT_JINA, 'flat', force_rebuild=False)

# Jina HNSW
jina_hnsw = build_or_load_index(jina_img_emb, FAISS_HNSW_JINA, 'hnsw', force_rebuild=False)

# CLIP Flat
clip_flat = build_or_load_index(clip_img_emb, FAISS_FLAT_CLIP, 'flat', force_rebuild=False)

# CLIP HNSW
clip_hnsw = build_or_load_index(clip_img_emb, FAISS_HNSW_CLIP, 'hnsw', force_rebuild=False)

print('✅ Все индексы построены/загружены')

## 5. Демо поиска

### 5.1 Запрос: 'a dog on the beach' (Jina)

In [ ]:
results, search_time = search_images(
    query='a dog on the beach',
    index_path=FAISS_FLAT_JINA,
    image_paths=image_paths,
    captions=captions,
    model_name='jina',
    top_k=5,
)

print(f"⏱️ Поиск за {search_time*1000:.1f} мс")
display_results('a dog on the beach', results)

### 5.2 Запрос: 'a red car' (CLIP)

In [ ]:
results, search_time = search_images(
    query='a red car',
    index_path=FAISS_FLAT_CLIP,
    image_paths=image_paths,
    captions=captions,
    model_name='clip',
    top_k=5,
)

print(f"⏱️ Поиск за {search_time*1000:.1f} мс")
display_results('a red car', results)

### 5.3 Запрос: 'people playing football' (Jina)

In [ ]:
results, search_time = search_images(
    query='people playing football',
    index_path=FAISS_FLAT_JINA,
    image_paths=image_paths,
    captions=captions,
    model_name='jina',
    top_k=5,
)

print(f"⏱️ Поиск за {search_time*1000:.1f} мс")
display_results('people playing football', results)

## 6. Оценка качества поиска

Метрики:
- **Precision@k** — доля релевантных в топ-k
- **Recall@k** — доля найденных релевантных из всех релевантных
- **mAP** — mean Average Precision (усреднение AP по всем запросам)

In [ ]:
%%time
# Оценка Jina
df_jina = evaluate_model(
    queries=TEST_QUERIES,
    captions=captions,
    image_paths=image_paths,
    index_path=FAISS_FLAT_JINA,
    model_name='jina',
    top_k_values=[1, 3, 5, 10],
)

print('📊 Jina v5-omni-nano — результаты:')
print(df_jina.to_string())

In [ ]:
%%time
# Оценка CLIP
df_clip = evaluate_model(
    queries=TEST_QUERIES,
    captions=captions,
    image_paths=image_paths,
    index_path=FAISS_FLAT_CLIP,
    model_name='clip',
    top_k_values=[1, 3, 5, 10],
)

print('📊 CLIP ViT-B/32 — результаты:')
print(df_clip.to_string())

### 6.1 Сравнение моделей

In [ ]:
comparison = compare_models(TEST_QUERIES, captions, image_paths)
print(comparison)

### 6.2 Сравнение индексов: Flat vs HNSW (Jina)

In [ ]:
idx_comparison = compare_index_types(TEST_QUERIES, captions, image_paths, model_name='jina')
print(idx_comparison)

## 7. Итоговые выводы

### Техническая реализация
- Реализован пайплайн: данные -> эмбеддинги -> индекс -> поиск
- Две модели: Jina v5-omni-nano (1.04B) и CLIP ViT-B/32 (150M)
- Два типа индексов: FAISS Flat (exact) и HNSW (ANN)
- Оптимизации под CPU: modality=vision, truncate_dim=256, batch_size=4

### Качество поиска
- Релевантные результаты на тестовых запросах
- Сравнение моделей по метрикам P@k, R@k, mAP
- Анализ trade-offs: точность vs скорость

### Эксперименты
- Jina vs CLIP: сравнение архитектур и размерностей
- Flat vs HNSW: сравнение стратегий индексации
- Визуализация результатов поиска

### Production-ready фичи
- Docker-контейнеризация
- Streamlit веб-интерфейс
- Кэширование эмбеддингов и индексов
- Модульная архитектура (src/)